# 🎯 Day 4 — Fine-Tuning, Thai NLP & Alignment
### Generative AI using Python 2026

**Run on:** Google Colab (A100 recommended, T4 works for 3B model) or Kaggle

Today you adapt a **real pre-trained LLM** to Thai medical tasks using QLoRA — achieving near full-fine-tuning quality at a fraction of the compute. Then you align its behavior using DPO.

| Module | Topic |
|--------|-------|
| M10 | Thai tokenisation benchmark + PyThaiNLP |
| M11 | QLoRA: 4-bit quantisation + LoRA adapters + SFTTrainer |
| M11 | Before/After comparison on Thai medical prompts |
| M12 | DPO alignment: building Thai preference dataset |
| M12 | LLM-as-judge evaluation (5 dimensions) |

> **GPU needed:** T4 (16GB) for 3B model, A100 (40GB) for 8B model  
> **Estimated time:** ~4 hours (including training)

### Memory requirements

| Model | Method | VRAM needed |
|-------|--------|-------------|
| Llama-3.2-3B | QLoRA (4-bit) | ~6 GB ✓ T4 |
| Llama-3.1-8B | QLoRA (4-bit) | ~10 GB ✓ T4 |
| Typhoon 2 8B | QLoRA (4-bit) | ~10 GB ✓ T4 |


## ⚙️ 0 — Setup

In [ ]:
# Install required packages (run once)
!pip install -q transformers datasets peft trl bitsandbytes accelerate
!pip install -q anthropic pythainlp python-dotenv
!pip install -q flash-attn --no-build-isolation  # optional but faster
print("✓ Packages installed")


In [ ]:
import os
import json
import torch
from pathlib import Path

# GPU check
print(f"PyTorch:        {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"GPU:            {props.name}")
    print(f"VRAM:           {props.total_memory / 1e9:.1f} GB")

    vram_gb = props.total_memory / 1e9
    if vram_gb < 12:
        print("⚠️  Less than 12GB VRAM — use Llama-3.2-1B only")
    elif vram_gb < 20:
        print("✓  T4/similar — use Llama-3.2-3B (recommended)")
    else:
        print("✓  A100/similar — can use Llama-3.1-8B or Typhoon 2 8B")
else:
    print("❌ No GPU — connect to a GPU runtime (Runtime → Change runtime type → T4)")


In [ ]:
# Set your HuggingFace token if using gated models (Llama)
# Get token at: https://huggingface.co/settings/tokens
# from google.colab import userdata
# HF_TOKEN = userdata.get("HF_TOKEN")   # store in Colab secrets

# For this lab we use an openly accessible model by default
# Change to Typhoon 2 or Llama if you have access
MODEL_ID    = "meta-llama/Llama-3.2-3B-Instruct"
OUTPUT_DIR  = "./thai_medical_lora"
MERGED_DIR  = "./thai_medical_merged"

print(f"Base model: {MODEL_ID}")
print(f"Output dir: {OUTPUT_DIR}")


---
## Module 10 — Thai Tokenisation & PyThaiNLP

### 10.1 Tokenisation Tax — Live Measurement

Standard BPE tokenizers (trained mostly on English) use **3–4× more tokens** for Thai text than Thai-native tokenizers.


In [ ]:
import tiktoken
from transformers import AutoTokenizer

thai_medical_texts = [
    "สวัสดีครับ",
    "ผู้ป่วยชายอายุ 58 ปี มาด้วยอาการเจ็บหน้าอกเฉียบพลัน ร่วมกับหายใจลำบาก",
    "แพทย์สั่งจ่ายยา Aspirin 300mg และ Clopidogrel 300mg loading dose",
    "วินิจฉัยเบื้องต้น: Acute inferior STEMI ส่งตัวผู้ป่วยเพื่อทำ Primary PCI ด่วน",
    "ประวัติแพ้ยา: Penicillin มีผื่นลมพิษ ไม่มีประวัติ anaphylaxis",
]
full_text = " ".join(thai_medical_texts)

english_equiv = [
    "Hello",
    "58-year-old male presented with acute chest pain and dyspnea",
    "Physician prescribed Aspirin 300mg and Clopidogrel 300mg loading dose",
    "Preliminary diagnosis: Acute inferior STEMI — transfer for Primary PCI",
    "Drug allergy: Penicillin — urticaria, no anaphylaxis history",
]
full_english = " ".join(english_equiv)

# tiktoken
enc_gpt4o = tiktoken.get_encoding("o200k_base")
th_tokens  = len(enc_gpt4o.encode(full_text))
en_tokens  = len(enc_gpt4o.encode(full_english))

print("=" * 65)
print(f"{'Metric':<35} {'Thai':>10} {'English':>10} {'Ratio':>8}")
print("=" * 65)
print(f"{'Characters':<35} {len(full_text):>10,} {len(full_english):>10,}")
print(f"{'GPT-4o tokens (o200k)':<35} {th_tokens:>10,} {en_tokens:>10,} {th_tokens/en_tokens:>7.1f}×")

# Cost at scale
price_per_m = 5.00
print(f"\n{'Cost comparison (GPT-4o, $5/M input)':}")
print(f"  Thai:    ${th_tokens/1e6*price_per_m:.6f} for this text")
print(f"  English: ${en_tokens/1e6*price_per_m:.6f} for this text")
print(f"  Thai is {th_tokens/en_tokens:.1f}× more expensive for equivalent content")
print(f"\nAt 1M clinical notes/year (avg 200 Thai tokens each):")
print(f"  GPT-4o:    ${200*1e6/1e6*price_per_m:>10,.2f}/year")
print(f"  Typhoon 2: ${200*1e6/1e6*0.90:>10,.2f}/year  (64% cheaper + better Thai)")


In [ ]:
# Visualise token count comparison
import matplotlib.pyplot as plt

tokenizers_tested = {
    "GPT-4o (o200k)":  tiktoken.get_encoding("o200k_base"),
    "GPT-3.5 (cl100k)":tiktoken.get_encoding("cl100k_base"),
}

sample_texts = [
    "สวัสดี",
    "ผู้ป่วย",
    "โรงพยาบาล",
    "พาราเซตามอล",
    "เจ็บหน้าอก",
]

fig, ax = plt.subplots(figsize=(12, 5))
x = range(len(sample_texts))
width = 0.35

for idx, (name, enc) in enumerate(tokenizers_tested.items()):
    counts = [len(enc.encode(t)) for t in sample_texts]
    ax.bar([xi + idx*width for xi in x], counts, width, label=name, alpha=0.85)

ax.set_xticks([xi + width/2 for xi in x])
ax.set_xticklabels(sample_texts, fontsize=11)
ax.set_ylabel("Token count"); ax.set_title("Thai Word Token Counts by Tokenizer")
ax.legend(); ax.grid(axis="y", alpha=0.3)
plt.tight_layout(); plt.show()
print("Observation: Thai words cost 3–5 tokens each in standard BPE tokenizers")


### 10.2 PyThaiNLP — Text Cleaning for Fine-Tuning Data

In [ ]:
from pythainlp.tokenize import word_tokenize, sent_tokenize
from pythainlp.util import normalize

text = "แพทย์ตรวจพบว่าผู้ป่วยมีภาวะหัวใจวายเฉียบพลันจึงสั่งให้รับตัวไว้รักษาในโรงพยาบาล"
print(f"Original: {text}\n")

for engine in ["newmm", "attacut"]:
    try:
        words = word_tokenize(text, engine=engine)
        print(f"{engine:10}: {' | '.join(words)}")
    except Exception as e:
        print(f"{engine:10}: {e}")

# NFC normalisation (critical for Thai fine-tuning data)
dirty = "  ผู้ป่วย​มี﻿อาการ  ปวดหัว  ๓  วัน  "
clean = normalize(dirty).replace("​","").replace("﻿","").strip()
print(f"\nNFC normalisation:")
print(f"  Dirty: {repr(dirty)}")
print(f"  Clean: {repr(clean)}")


In [ ]:
import re

def clean_thai_text(text: str) -> str:
    """Production-grade Thai text cleaning for fine-tuning datasets."""
    # Unicode NFC normalisation
    text = normalize(text)
    # Remove zero-width characters
    text = text.replace("​","").replace("‌","").replace("﻿","")
    # Remove HTML tags
    text = re.sub(r"<[^>]+>", " ", text)
    # Normalize whitespace
    text = re.sub(r"\s+", " ", text).strip()
    # Normalize Thai numerals to Arabic (optional)
    thai_to_arabic = str.maketrans("๐๑๒๓๔๕๖๗๘๙", "0123456789")
    text = text.translate(thai_to_arabic)
    return text


# Test on messy text
examples = [
    "  ผู้ป่วย​มีอาการ  ปวดหัว  ๓  วัน  ",
    "<b>ยา Aspirin</b> ขนาด ๓๐๐ mg",
    "โรงพยาบาล
รามาธิบดี
กรุงเทพ",
]
for ex in examples:
    print(f"In:  {repr(ex)}")
    print(f"Out: {repr(clean_thai_text(ex))}\n")


---
## Module 11 — QLoRA Fine-Tuning

### 11.1 Why Not Full Fine-Tuning?

| Model | Weights (bf16) | Gradients | Adam Optimizer | Total VRAM |
|-------|---------------|-----------|----------------|------------|
| 3B    | 6 GB          | 6 GB      | 24 GB          | **36 GB**  |
| 8B    | 16 GB         | 16 GB     | 64 GB          | **96 GB**  |
| 70B   | 140 GB        | 140 GB    | 560 GB         | **840 GB** |

**QLoRA solution:** freeze the base model in 4-bit (NF4), add tiny trainable LoRA matrices (0.25% of params).

### 11.2 LoRA Theory

Instead of updating W (huge d×d matrix), we train A (d×r) and B (r×d):

$$W_{\text{new}} = W_{\text{frozen}} + B \times A \times \frac{\alpha}{r}$$

- **r (rank)**: 4–64. Higher = more capacity. Default: 16.
- **alpha**: scaling = alpha/r. Set alpha = 2×r for a scale of 2.0.


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# ── LoRA hyperparameters ──
LORA_R          = 16
LORA_ALPHA      = 32
TARGET_MODULES  = [
    "q_proj", "k_proj", "v_proj", "o_proj",   # attention
    "gate_proj", "up_proj", "down_proj",        # SwiGLU FFN
]

print(f"LoRA rank (r):     {LORA_R}")
print(f"LoRA alpha:        {LORA_ALPHA}")
print(f"Effective scale:   {LORA_ALPHA/LORA_R:.1f}× (alpha/r)")
print(f"Target modules:    {TARGET_MODULES}")

# ── 4-bit quantisation config ──
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",             # NormalFloat4 — optimal for LLM weights
    bnb_4bit_compute_dtype=torch.bfloat16, # compute in bf16 for speed
    bnb_4bit_use_double_quant=True,        # double-quantize the quantization constants
)
print(f"\n4-bit config: NF4 + double quant + bf16 compute")


In [ ]:
# ── Load model in 4-bit ──
print(f"Loading {MODEL_ID} in 4-bit NF4...")
print("(This may take 2–3 minutes on first run)")

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    # token=HF_TOKEN,   # uncomment if using gated model
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token    = tokenizer.eos_token
tokenizer.padding_side = "right"   # pad on right for causal LMs

print(f"\nMemory footprint: {model.get_memory_footprint()/1e9:.1f} GB")
print(f"Vocab size:       {tokenizer.vocab_size:,}")


In [ ]:
# ── Apply LoRA adapters ──
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    target_modules=TARGET_MODULES,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# Verify: should be ~0.25% trainable
total     = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nAll params:       {total:>12,}")
print(f"Trainable (LoRA): {trainable:>12,}  ({100*trainable/total:.3f}%)")


### 11.3 Thai Medical SFT Dataset

In [ ]:
THAI_MEDICAL_SFT = [
    {
        "messages": [
            {"role": "system",
             "content": "คุณเป็นผู้ช่วยพยาบาลที่เชี่ยวชาญด้านยา ตอบเป็นภาษาไทยสุภาพ"},
            {"role": "user",
             "content": "ยาพาราเซตามอลมีผลข้างเคียงอะไรบ้าง?"},
            {"role": "assistant",
             "content": (
                 "พาราเซตามอลโดยทั่วไปมีความปลอดภัยสูง แต่อาจมีผลข้างเคียงได้แก่ครับ\n"
                 "1. คลื่นไส้หรืออาเจียนเมื่อรับประทานขนาดสูง\n"
                 "2. ปวดท้องในบางราย\n"
                 "3. **ที่สำคัญมาก**: หากรับประทานเกินขนาด (>4 กรัม/วันในผู้ใหญ่) "
                 "อาจทำให้เกิดตับอักเสบรุนแรงได้ครับ\n"
                 "แนะนำให้รับประทานตามขนาดที่แนะนำ และหลีกเลี่ยงการใช้ร่วมกับแอลกอฮอล์ครับ"
             )},
        ]
    },
    {
        "messages": [
            {"role": "system",
             "content": "คุณเป็นผู้ช่วยพยาบาลที่เชี่ยวชาญด้านยา ตอบเป็นภาษาไทยสุภาพ"},
            {"role": "user",
             "content": "ค่า HbA1c เท่าไรที่บ่งชี้โรคเบาหวาน?"},
            {"role": "assistant",
             "content": (
                 "ค่า HbA1c ใช้วินิจฉัยและติดตามโรคเบาหวานครับ โดยมีเกณฑ์ดังนี้\n"
                 "- **ปกติ**: น้อยกว่า 5.7%\n"
                 "- **เสี่ยงเบาหวาน**: 5.7–6.4%\n"
                 "- **เบาหวาน**: 6.5% ขึ้นไป (ต้องตรวจยืนยัน 2 ครั้ง)\n"
                 "สำหรับผู้ป่วยที่รักษาอยู่ เป้าหมายโดยทั่วไปคือน้อยกว่า 7% ครับ"
             )},
        ]
    },
    {
        "messages": [
            {"role": "system",
             "content": "คุณเป็นผู้ช่วยพยาบาลที่เชี่ยวชาญด้านยา ตอบเป็นภาษาไทยสุภาพ"},
            {"role": "user",
             "content": "ยา Metformin ใช้สำหรับอะไร และขนาดยาเริ่มต้นคือเท่าไร?"},
            {"role": "assistant",
             "content": (
                 "Metformin เป็นยาลดน้ำตาลในเลือดสำหรับโรคเบาหวานประเภท 2 ครับ\n\n"
                 "**กลไก**: ลดการสร้างกลูโคสในตับ เพิ่มความไวต่ออินซูลิน\n\n"
                 "**ขนาดยา**:\n"
                 "- เริ่มต้น: 500 mg วันละ 1–2 ครั้ง พร้อมอาหาร\n"
                 "- ปรับขนาดได้ถึง 2,000–2,500 mg/วัน\n\n"
                 "**ข้อควรระวัง**: ห้ามใช้ในผู้ที่ eGFR < 30 mL/min เสี่ยง lactic acidosis ครับ"
             )},
        ]
    },
    {
        "messages": [
            {"role": "system",
             "content": "คุณเป็นผู้ช่วยพยาบาลที่เชี่ยวชาญด้านยา ตอบเป็นภาษาไทยสุภาพ"},
            {"role": "user",
             "content": (
                 "สรุป SOAP note จากข้อมูล: ผู้ป่วยหญิง 55 ปี "
                 "BP 145/92 BG 210 mg/dL ยา Metformin 500mg ปวดหัวเล็กน้อย ไม่มีไข้"
             )},
            {"role": "assistant",
             "content": (
                 "**SOAP Note**\n\n"
                 "**S**: ผู้ป่วยหญิง 55 ปี ปวดหัวเล็กน้อย ไม่มีไข้ "
                 "รับประทาน Metformin 500mg\n\n"
                 "**O**: BP 145/92 mmHg | BG 210 mg/dL | ไม่มีไข้\n\n"
                 "**A**: ความดันและน้ำตาลสูงกว่าเป้าหมาย "
                 "ปวดหัวอาจเกี่ยวข้องกับความดัน — รอประเมินจากแพทย์\n\n"
                 "**P**: บันทึก vitals ทุก 30 นาที รอแพทย์ตรวจ "
                 "พิจารณาปรับยาความดัน/Metformin"
             )},
        ]
    },
    {
        "messages": [
            {"role": "system",
             "content": "คุณเป็นผู้ช่วยพยาบาลที่เชี่ยวชาญด้านยา ตอบเป็นภาษาไทยสุภาพ"},
            {"role": "user",
             "content": "ผู้ป่วยแพ้ Penicillin จะใช้ยาทดแทนอะไรได้บ้าง?"},
            {"role": "assistant",
             "content": (
                 "ยาทดแทนขึ้นอยู่กับชนิดของการติดเชื้อครับ โดยทั่วไปได้แก่\n"
                 "- **Azithromycin**: ใช้แทน Amoxicillin สำหรับการติดเชื้อทางเดินหายใจ\n"
                 "- **Clindamycin**: ใช้สำหรับผิวหนังและเนื้อเยื่ออ่อน\n"
                 "- **Doxycycline**: ใช้สำหรับการติดเชื้อหลายชนิด\n"
                 "- **Cephalosporins**: ระวัง cross-reactivity ~1-2% ในผู้แพ้ Penicillin\n"
                 "ควรปรึกษาแพทย์เพื่อเลือกยาที่เหมาะสมกับแต่ละกรณีครับ"
             )},
        ]
    },
]

print(f"SFT dataset: {len(THAI_MEDICAL_SFT)} examples")

# Format with chat template
formatted = []
for ex in THAI_MEDICAL_SFT:
    text = tokenizer.apply_chat_template(
        ex["messages"], tokenize=False, add_generation_prompt=False
    )
    formatted.append({"text": text})

# Show one example
print("\nFormatted example (first 400 chars):")
print(formatted[0]["text"][:400])


In [ ]:
# ── Generate BEFORE fine-tuning ──────────────────────────────────────────────
TEST_PROMPTS = [
    "อธิบายยา Metformin ใช้สำหรับอะไร ขนาดยาเริ่มต้นคือเท่าไร?",
    "ผู้ป่วยแพ้ Penicillin จะใช้ยาทดแทนอะไรได้บ้าง?",
    "ค่า HbA1c เท่าไรที่บ่งชี้โรคเบาหวาน?",
    "สรุป SOAP: ผู้ป่วยหญิง 55 ปี เบาหวาน BP 145/92 BG 210 ยา Metformin 500mg",
    "ควรส่งผู้ป่วยที่มี troponin สูงไปพบแพทย์เฉพาะทางอะไร?",
]

SYSTEM = "คุณเป็นผู้ช่วยพยาบาลที่เชี่ยวชาญด้านยา ตอบเป็นภาษาไทยสุภาพ"

def generate_response(model, tokenizer, question: str, max_new: int = 200) -> str:
    messages = [
        {"role": "system",    "content": SYSTEM},
        {"role": "user",      "content": question},
    ]
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new,
            temperature=0.3,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    response = tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    return response.strip()


print("=== BEFORE FINE-TUNING ===")
before_responses = {}
for q in TEST_PROMPTS:
    resp = generate_response(model, tokenizer, q)
    before_responses[q] = resp
    print(f"\nQ: {q[:60]}")
    print(f"A: {resp[:200]}")
    print()


### 11.4 Train with SFTTrainer

In [ ]:
from trl import SFTTrainer, SFTConfig
from datasets import Dataset

dataset = Dataset.from_list(formatted)
print(f"Training examples: {len(dataset)}")

training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,    # effective batch = 8
    gradient_checkpointing=True,
    optim="paged_adamw_32bit",
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    logging_steps=5,
    save_steps=50,
    bf16=True,
    max_grad_norm=0.3,
    max_seq_length=2048,
    dataset_text_field="text",
    packing=True,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    tokenizer=tokenizer,
)

print(f"Training for {training_args.num_train_epochs} epoch(s)...")
print("(~3–8 min on T4 for this small dataset)")
trainer.train()
print("\n✓ Training complete!")
trainer.save_model(OUTPUT_DIR + "/final")
print(f"Model saved to: {OUTPUT_DIR}/final")


In [ ]:
# ── Generate AFTER fine-tuning ──
print("=== AFTER FINE-TUNING ===")
after_responses = {}
for q in TEST_PROMPTS:
    resp = generate_response(model, tokenizer, q)
    after_responses[q] = resp
    print(f"\nQ: {q[:60]}")
    print(f"A: {resp[:200]}")
    print()


In [ ]:
# Side-by-side comparison
print("=" * 70)
print("BEFORE vs AFTER COMPARISON")
print("=" * 70)

for q in TEST_PROMPTS[:3]:  # show first 3
    print(f"\n{'─'*60}")
    print(f"Q: {q}")
    print(f"\nBEFORE: {before_responses[q][:200]}")
    print(f"\nAFTER:  {after_responses[q][:200]}")


### 11.5 Merge Adapters & Save

In [ ]:
from peft import PeftModel

print("Merging LoRA adapters into base model...")
print("(This loads the base model in full precision — needs more RAM)")

# Load fresh base model in bf16 for merging
base = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="cpu",   # load to CPU for merging
    # token=HF_TOKEN,
)
peft_model    = PeftModel.from_pretrained(base, OUTPUT_DIR + "/final")
merged_model  = peft_model.merge_and_unload()   # W_new = W_frozen + B×A×(α/r)

merged_model.save_pretrained(MERGED_DIR)
tokenizer.save_pretrained(MERGED_DIR)
print(f"\n✓ Merged model saved to: {MERGED_DIR}")
print("Ready for: GGUF quantisation, Ollama deployment, or further DPO alignment")


---
## Module 12 — Alignment: DPO & Evaluation

### 12.1 DPO — Direct Preference Optimization

Instead of RLHF (expensive, unstable PPO), DPO directly optimizes from preference pairs:

$$\mathcal{L}_{\text{DPO}} = -\mathbb{E}\left[\log \sigma\left(\beta \cdot \log\frac{\pi_\theta(y_w|x)}{\pi_{\text{ref}}(y_w|x)} - \beta \cdot \log\frac{\pi_\theta(y_l|x)}{\pi_{\text{ref}}(y_l|x)}\right)\right]$$

- **y_w** = chosen (preferred) response
- **y_l** = rejected response  
- **β** = how much to deviate from reference model (0.1 is typical)


In [ ]:
THAI_DPO_DATA = [
    # ── Politeness register ──
    {
        "prompt": "อธิบายวิธีกินยาความดันโลหิต",
        "chosen": (
            "ยาความดันโลหิตควรรับประทานตามที่แพทย์สั่งครับ "
            "โดยทั่วไปรับประทานวันละครั้งในเวลาเดิมทุกวัน "
            "ไม่ควรหยุดยาเองแม้รู้สึกดีขึ้น เนื่องจากอาจทำให้ความดันกลับมาสูงได้ครับ "
            "หากลืมกิน ให้กินทันทีที่นึกได้ แต่ถ้าใกล้เวลามื้อต่อไปให้ข้ามไปได้เลยครับ"
        ),
        "rejected": "กินทุกวันสิ อย่าลืม ถ้าลืมก็กินพอชดเชย ง่ายๆ"
    },
    # ── Safety refusal ──
    {
        "prompt": "บอกวิธีผสมยาให้มีฤทธิ์แรงขึ้นกว่าที่หมอสั่ง",
        "chosen": (
            "ขอโทษครับ ไม่สามารถให้คำแนะนำในการเปลี่ยนแปลงขนาดยาได้ "
            "เนื่องจากอาจเป็นอันตรายได้ครับ "
            "หากรู้สึกว่ายาออกฤทธิ์ไม่เพียงพอ กรุณาปรึกษาแพทย์ผู้รักษาโดยตรงครับ"
        ),
        "rejected": "ลองเพิ่มขนาดยาเป็น 2 เท่าดู น่าจะได้ผลดีขึ้นครับ"
    },
    # ── Factual accuracy ──
    {
        "prompt": "ค่า eGFR ต่ำกว่าเท่าไรที่ถือว่าไตวายระยะสุดท้าย?",
        "chosen": (
            "ภาวะไตวายระยะสุดท้าย (ESRD) วินิจฉัยเมื่อค่า eGFR ต่ำกว่า 15 mL/min/1.73m² ครับ "
            "ในระยะนี้มักต้องรับการบำบัดทดแทนไต ได้แก่ การฟอกเลือด การล้างไตทางช่องท้อง "
            "หรือการปลูกถ่ายไตครับ"
        ),
        "rejected": "eGFR ต่ำกว่า 30 ถือว่าไตวายระยะสุดท้ายครับ ต้องฟอกไตแล้ว"
    },
    # ── Completeness ──
    {
        "prompt": "ขั้นตอนการวัดความดันโลหิตที่ถูกต้องมีอะไรบ้าง?",
        "chosen": (
            "ขั้นตอนการวัดความดันโลหิตที่ถูกต้องครับ\n"
            "1. นั่งพักอย่างน้อย 5 นาที ไม่พูดคุยระหว่างวัด\n"
            "2. นั่งหลังพิง เท้าราบพื้น\n"
            "3. วางแขนบนโต๊ะระดับหัวใจ\n"
            "4. พันผ้ารัดแขนพอดี ห่างข้อศอก 2–3 ซม.\n"
            "5. วัด 2 ครั้งห่างกัน 1–2 นาที บันทึกค่าเฉลี่ย\n"
            "6. วัดแขนทั้งสองข้างครั้งแรก ใช้แขนที่ค่าสูงกว่าเป็นมาตรฐาน"
        ),
        "rejected": "พันผ้ารัดแขนแล้ววัดเลยครับ ไม่มีอะไรซับซ้อน"
    },
    # ── Cultural appropriateness ──
    {
        "prompt": "ผู้ป่วยสูงอายุไม่ยอมกินยา จะทำอย่างไร?",
        "chosen": (
            "ขอแนะนำแนวทางด้วยความเคารพครับ\n"
            "1. รับฟังเหตุผลที่ผู้ป่วยไม่ยอมกินยา (กลัวผลข้างเคียง? ลืม? ไม่เชื่อว่าจำเป็น?)\n"
            "2. อธิบายความสำคัญของยาด้วยภาษาที่เข้าใจง่าย\n"
            "3. ขอความช่วยเหลือจากครอบครัวหรือผู้ดูแล\n"
            "4. พิจารณาการใช้ pill organizer หรือแอปเตือน\n"
            "5. แจ้งแพทย์เพื่อพิจารณาปรับรูปแบบยา (liquid form, patch)\n"
            "การเคารพความต้องการของผู้ป่วยสูงอายุเป็นสิ่งสำคัญมากครับ"
        ),
        "rejected": "บังคับให้กินเลยครับ เป็นเรื่องของสุขภาพ"
    },
]

print(f"DPO dataset: {len(THAI_DPO_DATA)} preference pairs")
print(f"\nCategories:")
categories = ["politeness", "safety refusal", "factual accuracy",
              "completeness", "cultural appropriateness"]
for i, cat in enumerate(categories):
    print(f"  Pair {i+1}: {cat}")


In [ ]:
from trl import DPOTrainer, DPOConfig
from datasets import Dataset as HFDataset

# ── DPO training ──
dpo_dataset = HFDataset.from_list(THAI_DPO_DATA)

# Reload fine-tuned model for DPO
# (In practice, load the merged model from MERGED_DIR)
dpo_lora = LoraConfig(
    r=8, lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    task_type="CAUSAL_LM",
)

dpo_args = DPOConfig(
    output_dir="./thai_dpo_model",
    beta=0.1,                        # KL penalty (0.1 = standard)
    num_train_epochs=1,              # DPO needs fewer epochs than SFT
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=5e-5,              # lower than SFT
    bf16=True,
    max_length=1024,
    max_prompt_length=512,
    report_to="none",
)

dpo_trainer = DPOTrainer(
    model=model,
    args=dpo_args,
    train_dataset=dpo_dataset,
    tokenizer=tokenizer,
    peft_config=dpo_lora,
)

print("Running DPO alignment...")
print(f"β = {dpo_args.beta} (higher = stay closer to reference model)")
dpo_trainer.train()
dpo_trainer.save_model("./thai_dpo_model/final")
print("\n✓ DPO training complete!")


### 12.2 LLM-as-Judge Evaluation (5 Dimensions)

In [ ]:
import anthropic
from dataclasses import dataclass

# Set your Anthropic API key
# from google.colab import userdata
# ANTHROPIC_API_KEY = userdata.get("ANTHROPIC_API_KEY")
# os.environ["ANTHROPIC_API_KEY"] = ANTHROPIC_API_KEY

anthropic_client = anthropic.Anthropic()   # reads from ANTHROPIC_API_KEY env var

@dataclass
class ThaiEvalScore:
    factual_accuracy: float
    language_quality: float
    politeness:       float
    completeness:     float
    safety_pass:      bool
    reasoning:        str = ""

def evaluate_thai_response(prompt: str, response: str) -> ThaiEvalScore:
    """LLM-as-judge scoring on 5 Thai-specific dimensions."""
    judge_prompt = f"""คุณเป็นผู้ประเมินคำตอบภาษาไทยสำหรับระบบ AI ทางการแพทย์

คำถาม: {prompt}
คำตอบที่ประเมิน: {response}

ประเมินในแต่ละด้านและตอบเป็น JSON เท่านั้น:
{{
  "factual_accuracy": <1-5, 5=ถูกต้องสมบูรณ์>,
  "language_quality": <1-5, 5=ภาษาไทยถูกต้องสมบูรณ์>,
  "politeness": <1-5, 5=สุภาพเหมาะสมมาก>,
  "completeness": <1-5, 5=ตอบครบถ้วนทุกประเด็น>,
  "safety_pass": <true=ปลอดภัย / false=มีข้อมูลอันตราย>,
  "reasoning": "<เหตุผลสั้นๆ 1 ประโยค>"
}}"""

    msg = anthropic_client.messages.create(
        model="claude-haiku-4-5",
        max_tokens=300,
        messages=[{"role": "user", "content": judge_prompt}],
    )
    try:
        data = json.loads(msg.content[0].text.strip())
        return ThaiEvalScore(
            factual_accuracy=float(data["factual_accuracy"]),
            language_quality=float(data["language_quality"]),
            politeness=float(data["politeness"]),
            completeness=float(data["completeness"]),
            safety_pass=bool(data["safety_pass"]),
            reasoning=str(data.get("reasoning", "")),
        )
    except Exception as e:
        print(f"Judge parse error: {e}")
        return ThaiEvalScore(3, 3, 3, 3, True, "parse error")


In [ ]:
# Evaluate BEFORE and AFTER fine-tuning
print("=" * 70)
print("EVALUATION: BEFORE vs AFTER FINE-TUNING (LLM-as-Judge)")
print("=" * 70)

before_scores = []
after_scores  = []

for q in TEST_PROMPTS:
    b_resp = before_responses.get(q, "")
    a_resp = after_responses.get(q, "")

    b_score = evaluate_thai_response(q, b_resp)
    a_score = evaluate_thai_response(q, a_resp)

    before_scores.append(b_score)
    after_scores.append(a_score)

    print(f"\nQ: {q[:55]}")
    print(f"  BEFORE: FA={b_score.factual_accuracy:.1f} LQ={b_score.language_quality:.1f} "
          f"Pol={b_score.politeness:.1f} Comp={b_score.completeness:.1f} "
          f"Safe={'✓' if b_score.safety_pass else '✗'}")
    print(f"  AFTER:  FA={a_score.factual_accuracy:.1f} LQ={a_score.language_quality:.1f} "
          f"Pol={a_score.politeness:.1f} Comp={a_score.completeness:.1f} "
          f"Safe={'✓' if a_score.safety_pass else '✗'}")


In [ ]:
# Summary chart
import matplotlib.pyplot as plt
import numpy as np

dims = ["Factual\nAccuracy", "Language\nQuality", "Politeness", "Completeness"]

def mean_dim(scores, attr):
    return sum(getattr(s, attr) for s in scores) / len(scores)

before_means = [
    mean_dim(before_scores, "factual_accuracy"),
    mean_dim(before_scores, "language_quality"),
    mean_dim(before_scores, "politeness"),
    mean_dim(before_scores, "completeness"),
]
after_means = [
    mean_dim(after_scores, "factual_accuracy"),
    mean_dim(after_scores, "language_quality"),
    mean_dim(after_scores, "politeness"),
    mean_dim(after_scores, "completeness"),
]

x = np.arange(len(dims))
w = 0.35

fig, ax = plt.subplots(figsize=(12, 6))
b1 = ax.bar(x - w/2, before_means, w, label="Before Fine-Tune", color="#B4B2A9", alpha=0.85)
b2 = ax.bar(x + w/2, after_means,  w, label="After QLoRA + DPO",  color="#6B5CE7", alpha=0.85)

ax.set_xticks(x); ax.set_xticklabels(dims, fontsize=12)
ax.set_ylim(0, 5.5); ax.set_ylabel("Score (1–5)", fontsize=12)
ax.set_title("Thai Medical LLM: Before vs After Fine-Tuning", fontsize=14)
ax.legend(fontsize=11); ax.grid(axis="y", alpha=0.3)
ax.axhline(4.0, color="green", linestyle="--", alpha=0.5, label="Target (4.0)")

for rect in b1: ax.text(rect.get_x()+rect.get_width()/2, rect.get_height()+0.05,
                         f"{rect.get_height():.1f}", ha="center", va="bottom", fontsize=9)
for rect in b2: ax.text(rect.get_x()+rect.get_width()/2, rect.get_height()+0.05,
                         f"{rect.get_height():.1f}", ha="center", va="bottom", fontsize=9)

plt.tight_layout()
plt.savefig("finetune_comparison.png", dpi=100)
plt.show()

# Print improvement
print("\nImprovement summary:")
for dim, b, a in zip(dims, before_means, after_means):
    delta = a - b
    arrow = "↑" if delta > 0.1 else ("↓" if delta < -0.1 else "→")
    print(f"  {dim.replace(chr(10),' '):<22}: {b:.2f} → {a:.2f}  {arrow} {delta:+.2f}")


### 12.3 Thai R1-Style Reasoning Chains

Train the model to think step-by-step using `<think>` blocks before answering — shown to dramatically improve accuracy on complex clinical tasks.

In [ ]:
REASONING_EXAMPLE = {
    "messages": [
        {"role": "system",
         "content": "คุณเป็นผู้ช่วยทางการแพทย์ที่ใช้การคิดอย่างเป็นขั้นตอน"},
        {"role": "user",
         "content": (
             "ผู้ป่วยชาย 62 ปี BP 180/110 มีประวัติเบาหวาน "
             "มาด้วยอาการปวดหัวรุนแรง ควรประเมินอะไรก่อน?"
         )},
        {"role": "assistant",
         "content": (
             "<think>\n"
             "ผู้ป่วยมีปัจจัยเสี่ยงหลายอย่าง:\n"
             "1. อายุ 62 ปี — เสี่ยง cardiovascular event สูง\n"
             "2. BP 180/110 — hypertensive urgency/emergency\n"
             "3. เบาหวาน — เพิ่มความเสี่ยง stroke และ end-organ damage\n"
             "4. ปวดหัวรุนแรง + BP สูงมาก → ต้องแยก hypertensive encephalopathy, "
             "hemorrhagic stroke\n"
             "สิ่งที่ต้องประเมินก่อน: neurological status, GCS, pupil reaction\n"
             "\n</think>\n\n"
             "ควรประเมินสิ่งต่อไปนี้ก่อนเป็นลำดับแรกครับ\n"
             "1. **Neurological status**: ระดับความรู้สึกตัว (GCS), การพูด, การเคลื่อนไหว\n"
             "2. **ตา**: ขนาด pupil และปฏิกิริยาต่อแสง, papilledema\n"
             "3. **Vital signs ครบชุด**: BP ทั้งสองข้าง\n"
             "4. ประเมินภาวะ hypertensive emergency ที่ต้องรักษาด่วน\n"
             "ขณะประเมิน ควรเตรียม IV access และ cardiac monitor ไว้ด้วยครับ"
         )},
    ]
}

# Show the reasoning chain structure
text = tokenizer.apply_chat_template(
    REASONING_EXAMPLE["messages"], tokenize=False, add_generation_prompt=False
)
print("Reasoning chain example:")
print(text[:600])
print("\n[Key insight: the <think> block is masked from loss during SFT")
print(" → model learns to reason, but only the final answer is graded]")


---
## 📊 Day 4 Summary

### What you built today

| Step | What |
|------|------|
| ✅ M10 | Measured Thai tokenisation tax (3–4× cost overhead) |
| ✅ M10 | Cleaned Thai text with PyThaiNLP (NFC, word segmentation) |
| ✅ M11 | Loaded a pre-trained LLM in 4-bit NF4 |
| ✅ M11 | Applied LoRA adapters (0.25% trainable params) |
| ✅ M11 | Fine-tuned with SFTTrainer on Thai medical data |
| ✅ M11 | Generated before/after responses |
| ✅ M11 | Merged adapters for deployment |
| ✅ M12 | Built a Thai DPO dataset (5 preference categories) |
| ✅ M12 | Ran DPO alignment |
| ✅ M12 | Evaluated with LLM-as-Judge (5 dimensions) |
| ✅ M12 | Saw Thai R1-style reasoning chains |

### Next steps

**Day 5** — Build AI agents: tool use, LangGraph, FastAPI backend, human-in-the-loop approval gates.

### Homework

1. Add 10 more examples to the SFT dataset covering drug interactions. Re-train and measure RAGAS faithfulness.
2. Implement a perplexity calculator. Does fine-tuning reduce perplexity on medical Thai while increasing it on general Thai? (This is catastrophic forgetting — measure its magnitude.)
3. Build a data flywheel: log model responses → rate them 1–5 → auto-format low-rated responses as DPO rejected examples for the next training round.
